# Premier League Home Advantage Analysis

This project analyses Premier League match results from the 2021/22 to 2025/26 seasons to investigate the strength of home advantage.

In [1]:
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

Matplotlib is building the font cache; this may take a moment.


In [2]:
data_dir = Path("../data")

files = {
    "2021/22": data_dir / "premier_league_2021_22.csv",
    "2022/23": data_dir / "premier_league_2022_23.csv",
    "2023/24": data_dir / "premier_league_2023_24.csv",
    "2024/25": data_dir / "premier_league_2024_25.csv",
    "2025/26": data_dir / "premier_league_2025_26.csv",
}

In [3]:
season_dataframes = []

for season, file_path in files.items():
    season_df = pd.read_csv(file_path)
    season_df["Season"] = season
    season_dataframes.append(season_df)

matches = pd.concat(season_dataframes, ignore_index=True)

In [4]:
print(matches.shape)
matches.head()

(1900, 163)


,Div,Date,Time,HomeTeam,AwayTeam,FTHG,FTAG,FTR,HTHG,HTAG,...,BMGMCA,BVCH,BVCD,BVCA,CLCH,CLCD,CLCA,LBCH,LBCD,LBCA
0,E0,13/08/2021,20:00,Brentford,Arsenal,2,0,H,1,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,E0,14/08/2021,12:30,Man United,Leeds,5,1,H,1,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,E0,14/08/2021,15:00,Burnley,Brighton,1,2,A,1,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,E0,14/08/2021,15:00,Chelsea,Crystal Palace,3,0,H,2,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,E0,14/08/2021,15:00,Everton,Southampton,3,1,H,0,1,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## Data selection

The source files contain match statistics and betting-market variables. This analysis retains only the fields required to evaluate home advantage.

In [5]:
required_columns = [
    "Season",
    "Date",
    "HomeTeam",
    "AwayTeam",
    "FTHG",
    "FTAG",
    "FTR"
]

matches = matches[required_columns].copy()

matches.head()

,Season,Date,HomeTeam,AwayTeam,FTHG,FTAG,FTR
0,2021/22,13/08/2021,Brentford,Arsenal,2,0,H
1,2021/22,14/08/2021,Man United,Leeds,5,1,H
2,2021/22,14/08/2021,Burnley,Brighton,1,2,A
3,2021/22,14/08/2021,Chelsea,Crystal Palace,3,0,H
4,2021/22,14/08/2021,Everton,Southampton,3,1,H


In [6]:
print(matches.shape)
print(matches.columns.tolist())

(1900, 7)
['Season', 'Date', 'HomeTeam', 'AwayTeam', 'FTHG', 'FTAG', 'FTR']


In [7]:
matches.groupby("Season").size()

Season
2021/22    380
2022/23    380
2023/24    380
2024/25    380
2025/26    380
dtype: int64

## Data cleaning and validation

The combined dataset is checked for missing values, incorrect data types, duplicate matches and invalid results before analysis.

In [9]:
matches.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1900 entries, 0 to 1899
Data columns (total 7 columns):
 #   Column    Non-Null Count  Dtype 
---  ------    --------------  ----- 
 0   Season    1900 non-null   object
 1   Date      1900 non-null   object
 2   HomeTeam  1900 non-null   object
 3   AwayTeam  1900 non-null   object
 4   FTHG      1900 non-null   int64 
 5   FTAG      1900 non-null   int64 
 6   FTR       1900 non-null   object
dtypes: int64(2), object(5)
memory usage: 104.0+ KB


In [10]:
matches.isna().sum()

Season      0
Date        0
HomeTeam    0
AwayTeam    0
FTHG        0
FTAG        0
FTR         0
dtype: int64

In [11]:
matches["Date"] = pd.to_datetime(
    matches["Date"],
    format="%d/%m/%Y"
)

In [12]:
matches.dtypes

Season              object
Date        datetime64[ns]
HomeTeam            object
AwayTeam            object
FTHG                 int64
FTAG                 int64
FTR                 object
dtype: object

In [13]:
duplicate_matches = matches.duplicated(
    subset=["Season", "Date", "HomeTeam", "AwayTeam"]
).sum()

print(f"Duplicate matches: {duplicate_matches}")

Duplicate matches: 0


In [14]:
matches["FTR"].value_counts()

FTR
H    839
A    607
D    454
Name: count, dtype: int64

In [15]:
valid_results = {"H", "D", "A"}
invalid_results = set(matches["FTR"].unique()) - valid_results

print(f"Invalid result codes: {invalid_results}")

Invalid result codes: set()


In [16]:
result_mismatches = matches[
    ((matches["FTHG"] > matches["FTAG"]) & (matches["FTR"] != "H")) |
    ((matches["FTHG"] == matches["FTAG"]) & (matches["FTR"] != "D")) |
    ((matches["FTHG"] < matches["FTAG"]) & (matches["FTR"] != "A"))
]

print(f"Score/result mismatches: {len(result_mismatches)}")

Score/result mismatches: 0


In [17]:
matches["GoalDifference"] = matches["FTHG"] - matches["FTAG"]

matches["Result"] = matches["FTR"].map({
    "H": "Home Win",
    "D": "Draw",
    "A": "Away Win"
})

In [18]:
matches["HomePoints"] = matches["FTR"].map({
    "H": 3,
    "D": 1,
    "A": 0
})

matches["AwayPoints"] = matches["FTR"].map({
    "H": 0,
    "D": 1,
    "A": 3
})

In [19]:
matches.head()

,Season,Date,HomeTeam,AwayTeam,FTHG,FTAG,FTR,GoalDifference,Result,HomePoints,AwayPoints
0,2021/22,2021-08-13,Brentford,Arsenal,2,0,H,2,Home Win,3,0
1,2021/22,2021-08-14,Man United,Leeds,5,1,H,4,Home Win,3,0
2,2021/22,2021-08-14,Burnley,Brighton,1,2,A,-1,Away Win,0,3
3,2021/22,2021-08-14,Chelsea,Crystal Palace,3,0,H,3,Home Win,3,0
4,2021/22,2021-08-14,Everton,Southampton,3,1,H,2,Home Win,3,0


In [20]:
print(matches.shape)
print(matches.isna().sum())

(1900, 11)
Season            0
Date              0
HomeTeam          0
AwayTeam          0
FTHG              0
FTAG              0
FTR               0
GoalDifference    0
Result            0
HomePoints        0
AwayPoints        0
dtype: int64


In [21]:
matches.describe()

,Date,FTHG,FTAG,GoalDifference,HomePoints,AwayPoints
count,1900,1900.000000,1900.000000,1900.000000,1900.000000,1900.000000
mean,2024-01-07 09:32:12.631578880,1.597368,1.329474,0.267895,1.563684,1.197368
min,2021-08-13 00:00:00,0.000000,0.000000,-8.000000,0.000000,0.000000
25%,2022-10-16 00:00:00,1.000000,0.000000,-1.000000,0.000000,0.000000
50%,2023-12-30 00:00:00,1.000000,1.000000,0.000000,1.000000,1.000000
75%,2025-03-15 00:00:00,2.000000,2.000000,1.000000,3.000000,3.000000
max,2026-05-24 00:00:00,9.000000,8.000000,9.000000,3.000000,3.000000
std,NaN,1.318038,1.204203,1.899896,1.330030,1.296690


### Cleaning summary

The dataset contains 1,900 Premier League matches across five complete seasons. No missing values, duplicate fixtures, invalid result codes or score-result inconsistencies were identified. Dates were converted to a standard datetime format, and additional variables were created for goal difference, readable match results and league points.